In [2]:
import sys
import os

# Add the parent directory's 'src' folder to the Python path
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

# Import the modular functions from data_preprocessing.py
from data_preprocessing import (
    load_fashion_mnist,
    combine_dataset,
    split_dataset,
    preprocess_dataset
)

# 1. Load original Fashion-MNIST data
x_train, y_train, x_test, y_test = load_fashion_mnist()

# 2. Combine datasets
X, y = combine_dataset(x_train, y_train, x_test, y_test)

# 3. Create 70/15/15 train/val/test splits
X_train, X_val, X_test, y_train, y_val, y_test = split_dataset(X, y)

# 4. Normalize and add channel dimensions
X_train, X_val, X_test, y_train, y_val, y_test = preprocess_dataset(
    X_train, X_val, X_test, y_train, y_val, y_test
)

print("Data preprocessed successfully!")
print(f"Training data shape: {X_train.shape}")
print(f"Validation data shape: {X_val.shape}")
print(f"Test data shape: {X_test.shape}")

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 3us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 20s 1us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 2us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 5s 1us/step
Data preprocessed successfully!
Training data shape: (49000, 28, 28, 1)
Validation data shape: (10500, 28, 28, 1)
Test data shape: (10500, 28, 28, 1)


In [3]:
from tensorflow.keras import layers, models

# Build Model B (optimized CNN with < 100k parameters)
model_b = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model_b.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Print summary to verify parameters are under 100k
model_b.summary()

/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,626 (92.29 KB)

 Trainable params: 23,626 (92.29 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
import os

# 1. Train the model using your preprocessed training and validation data
history = model_b.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_val, y_val)
)

# 2. Evaluate on the test set
test_loss, test_accuracy = model_b.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_accuracy:.4f}")

# 3. Create the required directories if they don't already exist
os.makedirs(os.path.join('..', 'models'), exist_ok=True)
os.makedirs(os.path.join('..', 'results', 'model_thayarathan'), exist_ok=True)

# 4. Save the trained model to the root 'models/' folder as Model_Thayarathan.keras
model_save_path = os.path.join('..', 'models', 'Model_Thayarathan.keras')
model_b.save(model_save_path)
print(f"Model saved successfully to {model_save_path}")

Epoch 1/10
766/766 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.8311 - loss: 0.4735 - val_accuracy: 0.8281 - val_loss: 0.4842
Epoch 2/10
766/766 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.8364 - loss: 0.4592 - val_accuracy: 0.8389 - val_loss: 0.4631
Epoch 3/10
766/766 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.8407 - loss: 0.4472 - val_accuracy: 0.8444 - val_loss: 0.4458
Epoch 4/10
766/766 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.8474 - loss: 0.4302 - val_accuracy: 0.8416 - val_loss: 0.4555
Epoch 5/10
766/766 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.8508 - loss: 0.4177 - val_accuracy: 0.8485 - val_loss: 0.4300
Epoch 6/10
766/766 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.8531 - loss: 0.4114 - val_accuracy: 0.8530 - val_loss: 0.4229
Epoch 7/10
766/766 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.8580 - loss: 0.3976 - val_accuracy: 0.8665 - val_loss: 0.3902
Epoch 8/10
766/766 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - accuracy: 0.8590 - loss: 0.3927 - val_acc